# NB07 – Mục 5.4. Phân tích nguyên nhân hao hụt (bản đầy đủ)

Chạy **Runtime → Run all** một lần. Khi hiện nút *Choose Files*, chọn `NB07_5_3.zip`.

Thứ tự: (0) tìm dữ liệu → (1) nền 5.3 (variance_weekly) → (2) 5.4.1 Spearman + scatter → (3) 5.4.2 Z-score + biểu đồ → (4) hiển thị & tải kết quả.

## 0. Thiết lập môi trường

In [ ]:
import os, sys, zipfile
from pathlib import Path
NEED = ['sales_clean.csv','waste_log.csv','inventory_weekly.csv','bom.csv','ingredient.csv']
def _ok(p): return all((Path(p)/'input'/f).exists() for f in NEED)
def _search(root, depth=6):
    root=str(root)
    if not os.path.isdir(root): return None
    base=root.rstrip(os.sep).count(os.sep)
    for d,dirs,_ in os.walk(root):
        if d.count(os.sep)-base>=depth: dirs[:]=[]; continue
        dirs[:]=[x for x in dirs if not x.startswith('.git') and x not in ('__pycache__','sample_data')] if '/.shortcut' not in d else dirs
        if os.path.basename(d)=='NB07_5_3' and _ok(d): return d
    return None
def _find():
    cwd=Path.cwd()
    for p in [cwd,*cwd.parents[:2]]:
        if _ok(p): return str(p)
        if _ok(p/'NB07_5_3'): return str(p/'NB07_5_3')
    return None
root=_find()
if root is None and 'google.colab' in sys.modules or (root is None and os.path.isdir('/content')):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e: print('Không mount được Drive:',e)
    for r in ['/content/drive/MyDrive','/content/drive/.shortcut-targets-by-id','/content/drive/Shareddrives','/content']:
        root=_search(r)
        if root: break
if root is None and os.path.isdir('/content'):
    print('Không thấy thư mục NB07_5_3 trên Drive. Hãy chọn file NB07_5_3.zip để tải lên.')
    from google.colab import files
    up=files.upload()
    for n in up:
        if n.lower().endswith('.zip'):
            with zipfile.ZipFile(n) as z: z.extractall('/content')
    root=_search('/content')
if root is None:
    raise FileNotFoundError('Không tìm thấy thư mục NB07_5_3 (cần input/ với: '+', '.join(NEED)+'). Hãy chạy notebook trong thư mục đó hoặc tải zip lên.')
os.chdir(root)
print('Thư mục làm việc:',Path.cwd())
miss=[f for f in NEED if not (Path('input')/f).exists()]
assert not miss, f'Thiếu file trong input/: {miss}'
try:
    import scipy
except ImportError:
    get_ipython().run_line_magic('pip','install -q scipy')
print('Đủ dữ liệu đầu vào.')


## 1. Nền từ mục 5.3 (tạo variance_weekly.csv dùng cho 5.4)

In [ ]:
# Tạo processed/abc_revenue.csv và charts/abc_revenue.png nếu chưa có (ABC theo doanh thu món: A 0-80%, B 80-95%, C 95-100%)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
P=Path.cwd(); (P/'processed').mkdir(exist_ok=True); (P/'charts').mkdir(exist_ok=True)
if not (P/'processed'/'abc_revenue.csv').exists():
    s=pd.read_csv(P/'input'/'sales_clean.csv')
    r=s.groupby(['product_id','product_detail'],as_index=False).agg(revenue_usd=('revenue','sum'),quantity_sold=('transaction_qty','sum')).sort_values('revenue_usd',ascending=False)
    r['cumulative_pct']=100*r.revenue_usd.cumsum()/r.revenue_usd.sum()
    prev=r.cumulative_pct-100*r.revenue_usd/r.revenue_usd.sum()
    r['abc_class']=np.select([prev<80,prev<95],['A','B'],default='C')
    r.to_csv(P/'processed'/'abc_revenue.csv',index=False,encoding='utf-8-sig')
    fig,ax=plt.subplots(figsize=(12,5)); xs=np.arange(len(r)); ax.bar(xs,r.revenue_usd); ax.set_ylabel('USD')
    ax.set_xticks(xs[::5]); ax.set_xticklabels(r.product_detail.iloc[::5],rotation=65,ha='right',fontsize=7)
    ay=ax.twinx(); ay.plot(xs,r.cumulative_pct,color='darkorange',lw=2); ay.axhline(80,color='gray',ls='--'); ay.axhline(95,color='gray',ls=':'); ay.set_ylim(0,105)
    ax.set_title('Pareto doanh thu theo sản phẩm'); fig.tight_layout(); fig.savefig(P/'charts'/'abc_revenue.png',dpi=170); plt.close(fig)
    print('Đã tạo abc_revenue.csv:',len(r),'sản phẩm')
else:
    print('abc_revenue.csv đã có, bỏ qua.')


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
P=Path.cwd(); I=P/'input'; O=P/'processed'; C=P/'charts'
for d in (I,O,C):d.mkdir(exist_ok=True)
inv=pd.read_csv(I/'inventory_weekly.csv'); waste=pd.read_csv(I/'waste_log.csv')
keys=['store_id','ingredient_id','week_start']
assert not inv.duplicated(keys).any()
w=waste.groupby(keys,as_index=False).agg(waste_qty=('waste_qty','sum'),waste_log_value_usd=('waste_value_usd','sum'))
a=inv.merge(w,on=keys,how='left',validate='one_to_one');a[['waste_qty','waste_log_value_usd']]=a[['waste_qty','waste_log_value_usd']].fillna(0)
assert np.allclose(a.recorded_waste,a.waste_qty,atol=.001)
assert np.allclose(a.opening_qty+a.receipt_qty-a.closing_qty,a.actual_usage,atol=.001)
a['variance_qty']=a.actual_usage-a.theoretical_usage
a['variance_rate_pct']=np.where(a.theoretical_usage>0,100*a.variance_qty/a.theoretical_usage,np.nan)
a['waste_rate_pct']=np.where(a.actual_usage>0,100*a.waste_qty/a.actual_usage,np.nan)
a['unexplained_shrinkage_qty']=a.variance_qty-a.waste_qty
a['variance_value_usd']=a.variance_qty*a.unit_cost_usd
a['waste_value_usd']=a.waste_qty*a.unit_cost_usd
a['unexplained_shrinkage_value_usd']=a.unexplained_shrinkage_qty*a.unit_cost_usd
a['positive_shrinkage_value_usd']=a.unexplained_shrinkage_value_usd.clip(lower=0)
a['priority']=np.select([(a.variance_rate_pct>=8)&(a.unexplained_shrinkage_qty>0),(a.variance_rate_pct>=5)&(a.variance_rate_pct<8)&(a.unexplained_shrinkage_qty>0)],['Ưu tiên kiểm tra','Cảnh báo'],default='Bình thường')
a.to_csv(O/'variance_weekly.csv',index=False,encoding='utf-8-sig')
abc=pd.read_csv(O/'abc_revenue.csv')
# ABC by total positive excess usage value; preserve negative amounts separately in variance file.
b=a.groupby(['ingredient_id','ingredient_name','unit'],as_index=False).agg(positive_variance_value_usd=('variance_value_usd',lambda s:s.clip(lower=0).sum()),positive_shrinkage_value_usd=('positive_shrinkage_value_usd','sum'),net_variance_value_usd=('variance_value_usd','sum'),waste_value_usd=('waste_value_usd','sum'))
b=b.sort_values('positive_variance_value_usd',ascending=False).reset_index(drop=True)
total=b.positive_variance_value_usd.sum();b['cumulative_pct']=b.positive_variance_value_usd.cumsum()/total*100;b['previous_pct']=b.cumulative_pct-b.positive_variance_value_usd/total*100
b['abc_class']=np.select([b.previous_pct<80,b.previous_pct<95],['A','B'],default='C');b.to_csv(O/'abc_ingredient_variance.csv',index=False,encoding='utf-8-sig');b.head(10).to_csv(O/'top_10_ingredient_variance.csv',index=False,encoding='utf-8-sig')
def plot(d,val,name,title,file):
 d=d.sort_values(val,ascending=False).reset_index(drop=True);pct=d[val].cumsum()/d[val].sum()*100
 fig,ax=plt.subplots(figsize=(13,5.8));x=np.arange(len(d));ax.bar(x,d[val]);ax.set_ylabel('USD');ax.set_xticks(x);ax.set_xticklabels(d[name],rotation=70,ha='right',fontsize=7);ax.set_title(title)
 ax2=ax.twinx();ax2.plot(x,pct,color='darkorange',lw=2);ax2.axhline(80,color='gray',ls='--');ax2.axhline(95,color='gray',ls=':');ax2.set_ylim(0,105);ax2.set_ylabel('Tỷ trọng lũy kế (%)');fig.tight_layout();fig.savefig(C/file,dpi=180);plt.close(fig)
plot(b,'positive_variance_value_usd','ingredient_name','Pareto giá trị chênh lệch tiêu hao dương theo NVL','abc_ingredient_variance.png')
# Focused tests
s1=a[(a.store_location=="Hell's Kitchen")&(a.ingredient_id=='I008')&a.week_no.between(10,13)].copy()
s2=a[(a.store_location=='Astoria')&(a.ingredient_id=='I006')].copy()
s2other=a[(a.store_location!='Astoria')&(a.ingredient_id=='I006')].copy()
checks=pd.DataFrame([{'scenario':'a','rows':len(s1),'priority_rows':(s1.priority=='Ưu tiên kiểm tra').sum(),'variance_rate_min':s1.variance_rate_pct.min(),'variance_rate_max':s1.variance_rate_pct.max(),'waste_rate_min':s1.waste_rate_pct.min(),'waste_rate_max':s1.waste_rate_pct.max(),'unexplained_qty':s1.unexplained_shrinkage_qty.sum(),'unexplained_usd':s1.unexplained_shrinkage_value_usd.sum()}, {'scenario':'b','rows':len(s2),'warning_rows':(s2.priority=='Cảnh báo').sum(),'priority_rows':(s2.priority=='Ưu tiên kiểm tra').sum(),'variance_rate_weighted':100*s2.variance_qty.sum()/s2.theoretical_usage.sum(),'other_stores_weighted':100*s2other.variance_qty.sum()/s2other.theoretical_usage.sum()}]);checks.to_csv(O/'scenario_validation.csv',index=False,encoding='utf-8-sig')
print('Rows',len(a),'ABC',len(b),'total positive variance value',total)
print('Scenario tests:');print(checks.to_string(index=False));print('Top 10:');print(b[['ingredient_name','positive_variance_value_usd','abc_class']].head(10).to_string(index=False))
print('Summary:',a[['variance_value_usd','waste_value_usd','unexplained_shrinkage_value_usd','positive_shrinkage_value_usd']].sum().to_dict())


## 5.4.1. Tương quan Spearman giữa số giao dịch và hủy theo ca

In [ ]:
# ===== 5.4.1 Spearman: số giao dịch trong ca vs giá trị hủy ghi nhận cùng ca =====
from scipy.stats import spearmanr
sales=pd.read_csv(I/'sales_clean.csv',usecols=['transaction_id','store_location','week_start','shift'])
wl=pd.read_csv(I/'waste_log.csv')
k3=['store_location','week_start','shift']
tx=sales.groupby(k3,as_index=False).agg(n_transactions=('transaction_id','nunique'))
wv=wl.groupby(k3,as_index=False).agg(waste_value_usd=('waste_value_usd','sum'),waste_events=('waste_id','size'))
shift_df=tx.merge(wv,on=k3,how='outer').fillna({'waste_value_usd':0,'waste_events':0})
assert shift_df.n_transactions.notna().all(), 'Có ca có phiếu hủy nhưng không có giao dịch'
shift_df['week_start']=pd.to_datetime(shift_df.week_start)
first_w,last_w=shift_df.week_start.min(),shift_df.week_start.max()
def sp(d):
    r,p=spearmanr(d.n_transactions,d.waste_value_usd); return pd.Series({'n':len(d),'spearman_r':r,'p_value':p,'significant_0.05':p<0.05})
rows=[('Toàn bộ (cửa hàng × tuần × ca)',sp(shift_df)),
      ('Bỏ tuần 1 và tuần 27 (không đủ 7 ngày)',sp(shift_df[~shift_df.week_start.isin([first_w,last_w])]))]
for s,d in shift_df.groupby('store_location'): rows.append((s,sp(d)))
for s,d in shift_df.groupby('shift'): rows.append(('Ca '+s,sp(d)))
corr=pd.DataFrame({k:v for k,v in rows}).T.reset_index().rename(columns={'index':'pham_vi'})
corr.to_csv(O/'spearman_shift.csv',index=False,encoding='utf-8-sig'); shift_df.to_csv(O/'shift_tx_waste.csv',index=False,encoding='utf-8-sig')
print(corr.to_string(index=False))
# Scatter
r0,p0=spearmanr(shift_df.n_transactions,shift_df.waste_value_usd)
fig,ax=plt.subplots(figsize=(8.5,5.6))
for sh,col in zip(['Sáng','Chiều','Tối'],['#1f77b4','#ff7f0e','#2ca02c']):
    d=shift_df[shift_df['shift']==sh];ax.scatter(d.n_transactions,d.waste_value_usd,s=22,alpha=.65,label=f'Ca {sh}',color=col)
ax.set_xlabel('Số giao dịch trong ca (cửa hàng × tuần × ca)');ax.set_ylabel('Giá trị hủy ghi nhận cùng ca (USD)')
ax.set_title(f'Mật độ giao dịch và giá trị hủy theo ca (Spearman r = {r0:.3f}, p = {p0:.3g}, n = {len(shift_df)})',fontsize=10)
ax.legend();ax.grid(alpha=.25);fig.tight_layout();fig.savefig(C/'scatter_giao_dich_huy_theo_ca.png',dpi=180);plt.show();plt.close(fig)


## 5.4.2. Tuần và cửa hàng bất thường (Z-score)

In [ ]:
# ===== 5.4.2 Z-score của Variance Rate theo cửa hàng × tuần =====
v=a.copy(); v['theoretical_value_usd']=v.theoretical_usage*v.unit_cost_usd   # quy về USD để cộng các NVL khác đơn vị
sw=v.groupby(['store_location','week_no','week_start'],as_index=False).agg(days_observed=('days_observed','max'),variance_value_usd=('variance_value_usd','sum'),theoretical_value_usd=('theoretical_value_usd','sum'),waste_value_usd=('waste_value_usd','sum'),unexplained_value_usd=('unexplained_shrinkage_value_usd','sum'))
sw['variance_rate_pct']=100*sw.variance_value_usd/sw.theoretical_value_usd
g=sw.groupby('store_location').variance_rate_pct
sw['mean_store']=g.transform('mean'); sw['std_store']=g.transform(lambda s:s.std(ddof=1))
sw['z_score']=(sw.variance_rate_pct-sw.mean_store)/sw.std_store
sw['abnormal']=sw.z_score.abs()>2
sw.to_csv(O/'zscore_store_week.csv',index=False,encoding='utf-8-sig')
ab=sw[sw.abnormal].sort_values('z_score',key=abs,ascending=False)
ab.to_csv(O/'tuan_bat_thuong.csv',index=False,encoding='utf-8-sig')
print('Số tuần bất thường:',len(ab),'/',len(sw));print(ab[['store_location','week_no','week_start','days_observed','variance_rate_pct','z_score','unexplained_value_usd']].round(3).to_string(index=False))
# Đối chiếu kịch bản đã cài ở cấp cửa hàng và cấp NVL (kiểm chứng khả năng phát hiện)
print('Cửa hàng × tuần của kịch bản (a):');print(sw[(sw.store_location=="Hell's Kitchen")&sw.week_no.between(10,13)][['week_no','variance_rate_pct','z_score']].round(3).to_string(index=False))
v['z_series']=v.groupby(['store_location','ingredient_id']).variance_rate_pct.transform(lambda s:(s-s.mean())/s.std(ddof=1))
sc_a=v[(v.store_location=="Hell's Kitchen")&(v.ingredient_id=='I008')&v.week_no.between(10,13)][['week_no','variance_rate_pct','z_series']]
sc_b=v[(v.store_location=='Astoria')&(v.ingredient_id=='I006')]
print('Kịch bản (a) cấp NVL (Z theo chuỗi cửa hàng × NVL):');print(sc_a.round(3).to_string(index=False))
print('Kịch bản (b): số tuần |Z|>2 cấp NVL =',int((sc_b.z_series.abs()>2).sum()),'/',len(sc_b))
print('Số dòng cửa hàng×NVL×tuần có |Z|>2:',int((v.z_series.abs()>2).sum()),'/',len(v))
# Biểu đồ Z-score theo tuần
fig,ax=plt.subplots(figsize=(11,4.8))
for s,col in zip(sw.store_location.unique(),['#1f77b4','#ff7f0e','#2ca02c']):
    d=sw[sw.store_location==s];ax.plot(d.week_no,d.z_score,marker='o',ms=3.5,lw=1.2,label=s,color=col)
for y in (2,-2):ax.axhline(y,color='red',ls='--',lw=1)
ax.axhline(0,color='gray',lw=.7);ax.set_xlabel('Tuần');ax.set_ylabel('Z-score của Variance Rate');ax.set_title('Z-score Variance Rate theo cửa hàng × tuần (ngưỡng |Z| = 2)')
ax.legend();ax.grid(alpha=.25);fig.tight_layout();fig.savefig(C/'zscore_variance_rate.png',dpi=180);plt.show();plt.close(fig)


## Kết quả & tải về
**Lưu ý:** tương quan không chứng minh quan hệ nhân quả; kết quả là kiểm chứng khả năng phát hiện, không phải kết luận gian lận.

In [ ]:
# Hiển thị lại 2 hình và bảng kết quả mục 5.4
from IPython.display import Image, display
for f in ['scatter_giao_dich_huy_theo_ca.png','zscore_variance_rate.png']:
    print(f); display(Image(str(C/f)))
print('Bảng 5.4.1 - Spearman:'); display(corr)
print('Bảng 5.4.2 - Tuần bất thường (|Z|>2):'); display(ab[['store_location','week_no','week_start','days_observed','variance_rate_pct','z_score','unexplained_value_usd']].round(3))

In [ ]:
# Gói kết quả 5.4 để tải về máy (zip gồm charts/ và processed/)
import shutil
from pathlib import Path
out=Path('/content/NB07_5_4_ketqua'); shutil.rmtree(out,ignore_errors=True); (out/'charts').mkdir(parents=True); (out/'processed').mkdir()
for f in ['scatter_giao_dich_huy_theo_ca.png','zscore_variance_rate.png']: shutil.copy(C/f,out/'charts'/f)
for f in ['spearman_shift.csv','shift_tx_waste.csv','zscore_store_week.csv','tuan_bat_thuong.csv']: shutil.copy(O/f,out/'processed'/f)
shutil.make_archive('/content/NB07_5_4_ketqua','zip',out)
try:
    from google.colab import files; files.download('/content/NB07_5_4_ketqua.zip')
except Exception as e: print('Zip tại /content/NB07_5_4_ketqua.zip',e)